<a href="https://colab.research.google.com/github/AIforPhilanthropy/eduall/blob/main/learning_guides/EduAll_EDA_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EduAll: exploring the applicant pool before you decide

**Level 1 of computer-assisted decision making.** In this notebook the computer *describes* the 500 applicants: counts, distributions, comparisons. It does not read the applications, weigh criteria or decide for you. You stay the one who interprets.

**Why explore first?** Nobody can hold 500 applicants × 44 variables (22,000 values) in mind at once. A few well-chosen tables and plots show you the shape of the pool, and above all **where the hard trade-offs are**. Later, in the AnythingLLM workflow, you will ask the assistant to screen and compare applicants. The exploration here tells you *which questions are worth asking it*, and gives you numbers to check its answers against.

### How the exercises work
The exercises go from general (who applied?) to specific (which rules create which winners?). Each one has:
1. **Why it matters**: what this aspect of the data means for the decision.
2. **What to tell Gemini**: the information Gemini needs. You write the prompt yourself.
3. **An empty code cell**: click it, then click **Generate** (or open the **Gemini side panel**), type your request, read the code it proposes and run it.
4. **A reading cell**: write what you conclude.

Exercise 1 is solved for you, with the exact prompt we used.

### Talking to Gemini: the four things it always needs
- **The data:** the table is called `df`.
- **The variables:** the exact column names (they are given in each exercise).
- **The output:** a number, a table or a chart, and which chart.
- **The grouping or filter:** for example "for each `study_level`" or "only applicants with `gpa` below 2.5".

Before you trust a result: check that the code uses the right columns, that the number of rows makes sense (500, or the size of your filter), and that rerunning gives the same answer. If a cell fails, click **Explain error** and read the fix before accepting it.

## Step 0. Get the data

Run the two cells below. The first one copies the course repository from GitHub into Colab; the second loads the applicant table into `df`. Nothing to upload.

In [ ]:
# --- Get the data -------------------------------------------------------------------------------
import os, pathlib, subprocess
REPO_URL = "https://github.com/AIforPhilanthropy/eduall.git"
NOTEBOOK_DIR = "learning_guides"   # this notebook's folder
here = pathlib.Path.cwd()
if not any((p / "data" / "EduAll_500_Applications_Dataset.csv").exists() for p in [here, *here.parents]):
    if not pathlib.Path("/content/eduall").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/eduall"], check=True)
    target = pathlib.Path("/content/eduall") / NOTEBOOK_DIR
    os.chdir(target if target.exists() else pathlib.Path("/content/eduall"))
print("working directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)

here = pathlib.Path.cwd()
CSV = next(p / "data" / "EduAll_500_Applications_Dataset.csv" for p in [here, *here.parents]
           if (p / "data" / "EduAll_500_Applications_Dataset.csv").exists())
df = pd.read_csv(CSV)
print(f"{df.shape[0]} applicants x {df.shape[1]} variables")
df.head(3)

## Step 1. Know your variables

The 44 columns fall into six groups. Keep this list open: you will give these names to Gemini.

| Group | Columns |
|---|---|
| Identification | `student_id`, `application_date` |
| Demographic | `age`, `gender`, `country_origin`, `gni_tier`, `geographic_region`, `dependent_children`, `residency_permit` |
| Academic | `study_level`, `university`, `field_of_study`, `semesters_in_program`, `study_mode`, `planned_study_mode_2027`, `gpa`, `average_grade`, `courses_completed`, `publications`, `academic_standing` |
| Financial | `family_income_chf`, `family_household_income_chf`, `personal_income_chf`, `expected_personal_income_2027_chf`, `monthly_expenses_chf`, `housing_cost_chf`, `tuition_fees_per_year_chf`, `savings_chf`, `financial_need_chf`, `has_external_scholarship`, `external_scholarship_amount_chf` |
| Risk factors (0/1) | `risk_first_generation`, `risk_refugee_asylum`, `risk_disability`, `risk_housing_insecurity`, `risk_family_emergency`, `risk_language_barrier`, `risk_parent_student`, `risk_undocumented_status` |
| Resilience factors (0/1) | `resilience_volunteer_experience`, `resilience_work_experience`, `resilience_leadership_role`, `resilience_community_engagement`, `resilience_overcame_adversity` |

**Definitions that change decisions**
- `financial_need_chf` is the **current monthly** shortfall: monthly expenses − family support − personal income (minimum 0). It does **not** subtract scholarships (per year) or savings.
- `gni_tier`: income group of the country of origin; **1 = lowest income**, the Grant Call's priority countries.
- `academic_standing = "At Risk"` means a GPA below 2.5/4.0, the call's normal minimum.
- Maximum grant per year: bachelor CHF 8,000, master CHF 6,500, doctoral CHF 5,000. Total budget: CHF 500,000.

---
# Part A. Who applied?

## Exercise 1 (solved). The shape of the pool: study level and country income tier

**Why it matters.** The Grant Call sets a different maximum grant and an expected number of awards for each study level, and it gives priority to low-income countries (`gni_tier` = 1). Before any other question, you need to know how the 500 applicants split along these two axes: it tells you how much of the budget each group could absorb and whether the priority group is large or small.

**What we told Gemini.** The table name (`df`), the two variables (`study_level`, `gni_tier`), the outputs (a table with totals and a grouped bar chart) and how to arrange the chart.

**The exact prompt we used:**
> *Using the dataframe df, count the applicants by study_level and gni_tier in a table with row and column totals. Then draw a grouped bar chart of the same counts, with study level on the x-axis, one bar per gni_tier, and a title.*

**The code Gemini produced** (run it):

In [ ]:
counts = pd.crosstab(df["study_level"], df["gni_tier"], margins=True, margins_name="Total")
display(counts)

counts.drop(index="Total", columns="Total").plot(kind="bar", figsize=(8, 4), rot=0)
plt.title("Applicants by study level and income tier of the country of origin")
plt.xlabel("Study level")
plt.ylabel("Number of applicants")
plt.legend(title="gni_tier")
plt.tight_layout()
plt.show()

**Our reading.** Bachelor's (242) and master's (206) applicants dominate; doctoral applicants are few (52). Half of all applicants (249) come from tier 1 countries, the call's priority group, and they form the largest tier at every level. **Consequence for the decision:** "priority to low-income countries" alone cannot select the winners, because the priority group is larger than the 140–180 awards available. You will need further criteria, which the next exercises explore.

## Exercise 2 (optional). Where do applicants come from, and what do they study?

**Why it matters.** This is context, not a criterion: the Grant Call does not rank countries or fields. But you must know whether a few countries or universities dominate the pool, because a rule that looks neutral (for example "highest need first") can end up favouring one nationality. Nationality must not drive the decision by itself; knowing the composition lets you check that afterwards.

**What to tell Gemini.** Use `df`; for each of `country_origin`, `university` and `field_of_study`, show the 10 most frequent values with their counts, as horizontal bar charts.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

---
# Part B. Need: the core criterion

## Exercise 3. How much do applicants need?

**Why it matters.** Financial need is the heart of the Grant Call. Before ranking anyone, you should know what "typical" need looks like, how spread out it is, and how many applicants report **no** monthly shortfall at all. A zero does not automatically mean "no hardship" (savings and scholarships are not in this number, and income may change in 2027), so it is a case you will have to decide how to treat.

**What to tell Gemini.** Use `df` and the variable `financial_need_chf` (CHF per month). Ask for: a histogram, the median and mean, and the number of applicants whose value is exactly 0. Then ask for the median per `study_level`.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 4. Need against the budget: the scarcity problem

**Why it matters.** This exercise shows *why* you must prioritise. The budget is CHF 500,000 per year, but need is reported per **month**. Two comparisons make the scarcity concrete: (1) the pool's total need over a year against the budget; (2) how many applicants need more per month than the maximum grant could cover (CHF 8,000, 6,500 and 5,000 per year, i.e. about CHF 667, 542 and 417 per month for bachelor's, master's and doctoral students). If many applicants exceed the cap, "cover each person's full need" is not an option, and you must choose between giving many people part of what they need or fewer people more.

**What to tell Gemini.** Use `df`. (1) Compute the sum of `financial_need_chf` multiplied by 12 and compare it with 500,000. (2) Create a column with the monthly maximum grant for each `study_level` (the yearly maxima above divided by 12) and count, per `study_level`, how many applicants have `financial_need_chf` above that monthly maximum.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 5 (optional). Other resources: is the need column the whole story?

**Why it matters.** The Grant Call says awards normally do not exceed the documented shortfall *taking into account other scholarships and available savings*, and that the shortfall for 2027/28 uses the *expected* income. The `financial_need_chf` column does neither. If many high-need applicants have savings, a scholarship, or a lower expected income in 2027, ranking on `financial_need_chf` alone will be wrong for them. This is exactly the kind of case you will later ask the AnythingLLM workflow to cross-check against the application text.

**What to tell Gemini.** Use `df`. Count how many applicants have `savings_chf` above 0, how many have `has_external_scholarship` equal to 1, and how many have `expected_personal_income_2027_chf` lower than `personal_income_chf`. Then show the same three counts only for applicants whose `financial_need_chf` is above the median.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

---
# Part C. Merit and eligibility: the rules

## Exercise 6. Academic standing: how many fall below the rule?

**Why it matters.** The call asks for a GPA of normally at least 2.5/4.0, but allows exceptions if the applicant explains the circumstances. This is a **hard rule with a soft door**: you must decide whether "At Risk" applicants are excluded, considered case by case, or treated like everyone else. The decision is more important if these applicants also have high need, because excluding them would then remove some of the neediest people.

**What to tell Gemini.** Use `df`. Plot the distribution of `gpa` with a vertical line at 2.5. Count applicants per `academic_standing`. Then compare the median `financial_need_chf` of the "At Risk" group with the "Good" group.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 7 (optional). Time left to study

**Why it matters.** EduAll only funds study time from September 2027 onwards, and awards are pro-rated for students who finish during the grant year. A bachelor's student with 35 of 40 courses done needs much less support than one with 15. Some part-time students also plan to switch to full-time in 2027, which changes both their duration and their income.

**What to tell Gemini.** Use `df`. Create a column `courses_remaining` = required courses − `courses_completed`, where required courses are 40 for "Bachelor's" and 12 for "Master's" and "Doctoral" in `study_level` (never below 0). Show its distribution per `study_level`. Then make a table of `study_mode` against `planned_study_mode_2027`.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

---
# Part D. Vulnerability: risk and resilience

## Exercise 8. Risk factors: how common, and do they go with need?

**Why it matters.** The eight `risk_*` flags describe declared circumstances (first generation, refugee status, disability, housing insecurity…). You will have to decide whether they **raise priority**, only **break ties**, or stay out. Two traps: (1) a disadvantage counted twice, once through the risk flag and again through the need it already causes; (2) treating all flags as equivalent. Knowing how frequent each flag is, and whether more flags go with more need, helps you avoid both.

**What to tell Gemini.** Use `df`. Show the percentage of applicants with each column whose name starts with `risk_`, as a horizontal bar chart. Then create `n_risks`, the number of risk flags per applicant, and show the median `financial_need_chf` for each value of `n_risks`, with the number of applicants in each group.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 9 (optional). Special situations: pending permits and dependants

**Why it matters.** Some applicants have a residence permit still pending (`residency_permit` = "Pending", also flagged by `risk_undocumented_status`), and the call has specific rules for them. Others support children (`dependent_children`), but EduAll grants do not cover dependants' costs; a separate programme exists. These small groups need explicit decisions, and they are good candidates for case-by-case reading in the AnythingLLM workflow.

**What to tell Gemini.** Use `df`. Count applicants per `residency_permit`. Count applicants with `dependent_children` above 0. For both small groups, show the median `financial_need_chf` and `gpa`.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

---
# Part E. The trade-offs: what your rules will do

## Exercise 10. Need against merit: where your rule bites

**Why it matters.** If the neediest applicants were also the strongest students, there would be no dilemma. This exercise shows whether that is the case. Splitting applicants at the median GPA and the median need gives four groups. The two "mixed" groups (high need with lower GPA, and low need with higher GPA) are exactly where a need-first rule and a merit-first rule disagree, so they are where *your* decision matters most.

**What to tell Gemini.** Use `df`. Draw a scatter plot of `gpa` (x-axis) against `financial_need_chf` (y-axis), coloured by `gni_tier`, with dashed lines at the median of each variable. Then count the applicants in each of the four quadrants defined by those two medians.

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 11. Two candidate rules, two different winners

**Why it matters.** This is the bridge to decision making. Any eligibility rule selects a *pool*; different rules select different people. By comparing two simple rules you see their consequences before you commit to one: how many applicants each keeps, how much need it covers, and who it favours by study level and income tier. These are also the rules you can later give to the AnythingLLM workflow as **hard screens**, and the figures here let you check its answers.

- **Rule A, need first:** `financial_need_chf` above 0 and `gni_tier` equal to 1.
- **Rule B, merit first:** `financial_need_chf` above 0 and `gpa` at least 3.0.

**What to tell Gemini.** Use `df`. For Rule A and Rule B, show: the number of applicants selected, the total monthly `financial_need_chf`, and the counts by `study_level` and by `gni_tier`. Then count how many applicants are selected by both rules (match on `student_id`).

**Your reading:** _write two or three sentences here: what you see, and what it means for the allocation._

## Exercise 12. From exploration to questions for the AnythingLLM workflow

Look back at your readings. Write **three questions** you will ask the EduAll workflow assistant, and say for each whether it is a **hard screen** (a clear rule, such as "exclude applicants with zero need") or a **soft judgement** (a trade-off, such as "how should a GPA below 2.5 weigh against very high need?"). For each question, note the number from this notebook you will use to check its answer.

_Write your three questions here._

---
## What this exploration cannot do

It shows the shape of the pool, data problems and the size of candidate pools. It cannot read 500 personal statements, judge whether a story is credible, or tell you which rule is fair. That is the job of the next step, the AnythingLLM workflow, and above all of your own judgement.